In [1]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('analysis_formfactors')

enss=['b','c','d','e']
stouts_all=[5,7,10,13,15,20]

tftcphy_A20_conn=(0.8,0.2)
tftcphy_A20_discq=(0.6,0.2)
tftcphy_A20_discg=(0.6,0.3)

js_conn=['j+;conn','j-;conn']
js_discq=['j+;disc','js;disc','jc;disc'] 
js_gluon=[f'jg;stout{nst}' for nst in stouts_all]
js_disc=js_discq+js_gluon

In [3]:
ens2Njk={'b':725,'c':400,'d':493,'e':516}

path='data_aux/RCs_np.pkl'
with open(path,'rb') as f:
    ens2RCs_np_me=pickle.load(f)
ens2RCs_np={ens:{} for ens in enss}
for ens in enss:
    for key in ens2RCs_np_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs_np[ens][key]=yu.jackknife_pseudo(ens2RCs_np_me[ens][key],ens2RCs_np_me[ens][f'{key}_err']+1e-10,ens2Njk[ens])[:,0]

pdflabels=['HERAPDF2.0','ABMP16','CT18','MSHT20','NNPDF4.0','PDF4LHC21','JAM22','CJ22']
pdfsets=['HERAPDF20_NNLO_EIG','ABMP16_4_nnlo','CT18NNLO','MSHT20nnlo_as118','NNPDF40_nnlo_as_01180','PDF4LHC21_40_pdfas','JAM22-PDF_proton_nlo','CJ22']
label2j2me_A20=yu.load_pkl_reg('label2j2me_avgx',pathlabel='lhapdf')
label2j2me_A20={label:{j[:-2] if j.endswith(';+') else j:me for j,me in label2j2me_A20[label].items()} for label in label2j2me_A20.keys()}

key2phy_A20at0=yu.load_pkl_reg('step1/key2phy_A20',pathlabel='analysis_A20_syst')

In [5]:
cttejn_key2bare_A20=yu.load_pkl_reg(f'step1/cttejn_key2bare_A20',pathlabel='analysis_A20_syst')
key2bare={}; ens2Q2s={}
for iens,ens in enumerate(enss):
    n2qpp1s=list(set([key[-1] for key in cttejn_key2bare_A20.keys() if key[0]=='unequal' and key[3:4+1]==(ens,'j+;disc') and key[-1]!=(0,1,1) and yum.n2qpp12Q2(key[-1],ens)<=0.6]))
    Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
    ens2Q2s[ens]=Q2s
    
    for j in js_conn+js_disc:
        tt = tftcphy_A20_conn if j in js_conn else tftcphy_A20_discq if j in js_discq else tftcphy_A20_discg
        t=np.transpose([cttejn_key2bare_A20[('unequal',*tt,ens,j,n2qpp1)] for n2qpp1 in n2qpp1s])
        key2bare[(ens,j)]=t
        
yum.extendBare_avgx(key2bare)
key2phy=yum.bareRC2phy_avgx_np(key2bare,ens2RCs_np,ceQ=False)

In [13]:
t=key2phy_A20at0[('a=#_final','jv1')][:,0]
yu.jackme_un2str(t)

'0.1664(73)'

In [22]:
js=['jv1','jv2','jv3','jq;stout10','jg;stout10','jtot;stout10']
js2=[j.split(';')[0] for j in js]
fig,axs=yu.getFigAxs(len(js),1,sharey='row',sharex=True,Lrow=4,Lcol=16)
yu.addRowHeader(axs,js2)
# yu.addColHeader(axs,[yu.ens2label[ens] for ens in enss])
xunit=1; yunit=1
axs[0,0].set_xlim([-0.01,0.6])

for ij,j in enumerate(js):
    ax=axs[ij,0]
    
    t=key2phy_A20at0[('a=#_final',js2[ij])][:,0]
    mean,err=yu.jackme(t)
    plt_x=0*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
    ax.errorbar(plt_x,plt_y,plt_yerr,color='purple')
    
    for iens,ens in enumerate(enss):
        Q2s=ens2Q2s[ens]
        ys=key2phy[(ens,j)]
        mean,err=yu.jackme(ys)
        
        plt_x=(Q2s)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
        ax.errorbar(plt_x,plt_y,plt_yerr,color=yu.colors8[iens])
    
    a2_fit=np.concatenate([[yu.ens2a[ens]**2]*len(ens2Q2s[ens]) for ens in enss])
    Q2_fit=np.concatenate([ens2Q2s[ens] for ens in enss])
    y_fit=yu.superjackknife([key2phy[(ens,j)] for ens in enss])
    
    fitfunc_dipole_a2 = lambda Q2s,a2s,g,m: (g)/(1+Q2s/(m)**2)**2
    def fitfunc(pars):
        return fitfunc_dipole_a2(Q2_fit,a2_fit,*pars)
    pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_fit,pars0=[0.16,1],mask='uncorrelated')
    Q2_plt=np.arange(0,0.6,1/40)
    y_jk=[fitfunc_dipole_a2(Q2_plt,0,*pars) for pars in pars_jk]
    mean,err=yu.jackme(y_jk)
    plt_x=(Q2_plt)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
    ax.fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color='cyan',alpha=0.4,label=yu.un2str(plt_y[0],plt_yerr[0]))
    
    fitfunc_dipole_a2 = lambda Q2s,a2s,g,m,cg,cm: g/(1+Q2s/m**2)**2 + cg*a2s + cm*a2s*Q2s
    def fitfunc(pars):
        return fitfunc_dipole_a2(Q2_fit,a2_fit,*pars)
    pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_fit,pars0=[0.16,1,0,0],mask='uncorrelated')
    Q2_plt=np.arange(0,0.6,1/40)
    y_jk=[fitfunc_dipole_a2(Q2_plt,0,*pars) for pars in pars_jk]
    mean,err=yu.jackme(y_jk)
    plt_x=(Q2_plt)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
    ax.fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color='magenta',alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))

yu.finalizePlot('js_Q2')